|<img style="float:left;" src="images/usherb_transp.gif"> |Pierre Proulx, ing, professeur|
|:---|:---|
|Département de génie chimique et de génie biotechnologique |** GCH200-Phénomènes d'échanges I **|

## Sections  0, 1 et 2: Sections 1.0, 1.1 et 1.2: la viscosité des fluides et comment la viscosité affecte le mouvement des fluides. 
>#### Pour les deux premières sections, lire les pages 1-23 du document suivant:
[Chapitre 1, notes de cours sous format pdf](images/Chap1.pdf)
#### elles vous présentent le concept de viscosité et donnent des exemples de valeurs typiques des viscosités de certains fluides. 

# *

## Section 1.3, calcul de la viscosité réduite de fluides. La température, la pression et la viscosité sont normalisées en les divisant par la propriété critique. Le graphe 1.3-1 montre ces viscosités réduites.

> #### Le package "thermo" a été développé pour calculer les propriétés thermodynamiques et de transport en utilisant les méthodes les plus récentes de calcul. 

#### Voyons d'abord si le calcul des propriétés critiques est bien fait, comparons avec l'annexe E, rappelons-nous que le SI est pour Système International, donc Mètres, Kilogramme, Seconde, Joule, etc... Pas d'unités mixtes comme les poises, les atmosphères, les PSI, les livres-forces, et autres unités développées sans standard cohérent. Vérifiez dans l'annexe E si les calculs de CoolProp coincident bien avec les données de Transport Phenomena.

In [1]:
import thermo as th
CO2=th.Chemical('CO2')   # thermo utilise une approche objet
print('viscosité={:6.4e} Pa-s'.format(CO2.mug))    # viscosité du gaz       
print('pression et température en SI {:6.4e} Pa et {:5.2f} K'.format(CO2.P,CO2.T))      
      # L'objet CO2 est créé avec des températures et pressions initiales
CO2.T=400                # il est facile de la changer
print('viscosité={:6.4e} Pa-s'.format(CO2.mug))

viscosité=1.4915e-05 Pa-s
pression et température en SI 1.0132e+05 Pa et 298.15 K
viscosité=1.9635e-05 Pa-s


## La cellule qui suit est un exemple de l'utilisation des capacités interactives de Jupyter, n'hésitez pas à explorer ces capacités, elles peuvent rendre un notebook beaucoup plus intéressant et instructif

In [2]:
from ipywidgets import *
def viscosite(substance):
    a=th.Chemical(substance)
    a.T=300
    a.P=200000
    print('{:<4} viscosité={:10.4e} Pa-s à T={:8.2e} K et P={:8.2e} Pa'\
            .format(substance,a.mu,a.T,a.P))
    return
mu=interactive(viscosite,substance=['CO2','CO','argon','oxygen']);

In [3]:
display(mu)

interactive(children=(Dropdown(description='substance', options=('CO2', 'CO', 'argon', 'oxygen'), value='CO2')…

In [4]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
print(('{:<12}'*4).format('Fluide','Tc (K)','Pc (Pa)','muc (Pa*s)'))
print('-'*48)
for fluides in ['methane','ethane','propane','argon','N2','O2','CO2']:
    fluide=th.Chemical(fluides)
    CASRN=th.Chemical(fluides).CAS   # les fonctions Pc et Tc utilisent le nom standard CAS
    Tc=th.critical.Tc(CASRN)
    Pc=th.critical.Pc(CASRN)
    fluide.T=Tc
    fluide.P=Pc
    vc=fluide.mug
    print('{:<12}{:<12.3f}{:<12.3e}{:<12.3e}'.format(fluides,Tc,Pc,vc))

Fluide      Tc (K)      Pc (Pa)     muc (Pa*s)  
------------------------------------------------
methane     190.564     4.599e+06   7.351e-06   
ethane      305.322     4.872e+06   9.446e-06   
propane     369.890     4.251e+06   1.002e-05   
argon       150.687     4.863e+06   1.231e-05   
N2          126.192     3.396e+06   8.631e-06   
O2          154.581     5.043e+06   1.169e-05   
CO2         304.128     7.377e+06   1.520e-05   


## Section 1.4, calcul des viscosités de gaz à des pressions inférieures à 10% de la pression critique (CoolProp et thermo)

La viscosité individuelle peut être à partir de l'équation 1.4-14 en utilisant l'annexe E2.

>>> $
\begin{equation*}
  \boxed{ 
          \mu = 2.6693 \times 10^{-5} \frac{\sqrt{MT}}{\sigma^2 \Omega_{\mu}}
           }
  \end{equation*}
$ 


> Cependant, on a vu CoolProp qui effectue ce calcul, maintenant regardons comment un autre package python (thermo) utilise lui-même CoolProp pour calculer la viscosité à partir de 1.4-14, et utilise une syntaxe qui peut sembler plus facile et ajoute les calculs de mélange de gaz comme l'équation 1.4-15, 1.4-16 (Wilkes). 
>Pour un  mélange de gaz on peut approximer la viscosité en utilisant une pondération proposée par Wilkes en 1949.

>>> $
\begin{equation*}
  \boxed{
          \mu_{mix} = \sum_{i=0}^N \frac {x_i \mu_i} {\sum_{j=0}^N (x_j \phi_{ij})}
        }
\end{equation*}
$


avec 

>>> 
$ \begin{equation*}
  \boxed{
          \phi_{ij} = \frac {1} { \sqrt{8} }
                     \bigg ( 1+ \frac{M_i}{M_j} \bigg )^{-1/2}
                     \bigg [ 1+ \bigg ( \frac{\mu_i}{\mu_j} \bigg )^{1/2}
                     \bigg ( \frac{M_j}{M_i} \bigg )^{1/4}
                     \bigg ]^{2}
        }
        \end{equation*}
   $

 Voyons comment thermo résoud l'exemple 1.4-1. Il calcule les viscosités individuelles des 3 gaz en utilisant 1.4-14, 1.4-15 et 1.4-16. Le résultat est légèrement différent de celui de Transport Phenomena, pas moins précis.

In [5]:
# package thermo
import thermo as th
#
# regardons comment thermo calcule la viscosité d'un gaz  et d'un mélange
#
thermoMu   = th.Chemical('CO2',300,101325).mu
ExempleMu  = th.Mixture(['CO2','O2','N2'],zs=[0.133,0.039,0.828],T=293).mu

print('Avec thermo               : {:8.3e}'.format(thermoMu))
print("Calcul pour l'exemple 1.4-1 : {:8.3e}".format(ExempleMu))

Avec thermo               : 1.500e-05
Calcul pour l'exemple 1.4-1 : 1.755e-05


# *

## Section 1.5, calcul des viscosités de liquides (thermo)

>> Avec thermo, le calcul des propriétés des liquides est très simple et très naturel, par exemple pour l'eau liquide ou la vapeur d'eau à 50 degrés C et pression atmosphérique (on utilise toujours des unités du système international dans thermo, donc P=101325 Pascals = 1 atmosphère):

In [6]:
print(th.Chemical('water',273+50,101325).mug)
# une autre façon de faire le même travail, utilisant la nature objet du programme thermo
eau=th.Chemical('water',273+50,101325)
eau.mug

1.051717082743709e-05


1.051717082743709e-05

In [7]:
print(th.Chemical('water',273+50,101325).mul)
# ou puisqu'on a déjà défini l'objet eau de la classe Chemical, avec les T et P précédents
eau.mul   # on a pas besoin de print, car la dernière ligne de chaque cellulle est imprimée...

0.0005478953637518913


0.0005478953637518913

#### Comparons les valeurs obtenues avec thermo avec celles des équations proposées dans Transport Phenomena

In [8]:
# viscosité du benzène, exemple 1.5-1
print(th.Chemical('benzene', T=273+20, P=101325).mul)

0.0006482417530392874


#### Le résultat est bien meilleur, on peut le comparer avec le tableau 1.1-3, donc on utilisera donc thermo pour le calcul des viscosités des liquides. Continuons à comparer les valeurs prédites par thermo avec le tableau 1.1-3. Les méthodes utilisées par thermo pour calculer la viscosité des liquides sont beaucoup plus précises que celle proposée dans la section 1.5, alors on utilisera thermo.

In [12]:
print('***Tableau 1.1-3 Transport Phenomena, à comparer\n')
#Nom, Nom Thermo, temperature (°C), pression (kPa)
elements=[
    ['brome','bromine',   25,101.325],
    ['C2H5OH  0°C','C2H5OH',    0,101.325],
    ['C2H5OH 25°C','C2H5OH',    25,101.325],
    ['C2H5OH 50°C','C2H5OH',    50,101.325],
    ['Mercure','mercury', 20,101.325]
]

for elem in elements:
    mul=th.Chemical(elem[1], T=273+elem[2], P=elem[3]*1000).mul
    print(elem[0],mul)

***Tableau 1.1-3 Transport Phenomena, à comparer

brome 0.0009764345375901063
C2H5OH  0°C None
C2H5OH 25°C None
C2H5OH 50°C None
Mercure 0.0015682244256900425


# *

## Section 1.6- Calcul de la viscosité des suspensions.

En 1906, un jeune chercheur totalement inconnu de 27 ans propose une expression qui permet de calculer la viscosité de suspensions (diluées) de sphères solides dans un liquide. Cette expression se lit:

>>> $
\begin{equation*}
  \boxed{ 
          \frac {\mu_{eff}}{\mu_0}= 1 + \frac {5}{2}\phi
           }
  \end{equation*} (1.6-1)
$ 

En fait, il y avait une petite erreur de calcul, au lieu de 5/2 Einstein a donné 2 comme coefficient devant $\phi$. Il l'a corrigée quelques années plus tard, en 1911.

Cette équation a été utilisée comme base pour plusieurs travaux par la suite , celle de Mooney par exemple:

>>> $
\begin{equation*}
  \boxed{ 
          \frac {\mu_{eff}}{\mu_0}= exp \bigg(
                                      { \frac  {\frac {5}{2} \phi} {1-(\phi/\phi_0)} } \bigg )
           }
  \end{equation*} (1.6-2)
$ 


La cellule suivante effectue l'implémentation de la formule de Mooney et la compare avec un travail (Vand et al.) ou les auteurs ont lissé des résultats expérimentaux. C'est l'exercice 1B-3

In [13]:
# Exercice 1B3 en utilisant les capacités interactives pour mieux explorer le problème
# utilisez le contrôle de phi0 pour voir comment la courbe colle plus ou moins bien
# avec les résultats expérimentaux en rouge. Le coefficient phi0 est un paramètre "libre"!!
#
import sympy as sp
from IPython.display import *
from matplotlib import rcParams
sp.init_printing(use_latex=True)

def f1B3interactif(PHI0):  
    phi,phi0=sp.symbols('phi,phi_0')
    muV=1+2.5*phi+7.17*phi**2+16.2*phi**3
    muM=sp.Function('mu_M')(phi)
    muM=sp.exp((5./2.*phi)/(1-phi/phi0))
    muM=muM.subs(phi0,PHI0)
    rcParams['lines.linewidth'] = 3
    plt.rcParams['figure.figsize'] = 8,6   
    p1=sp.plot(muV,(phi,0,0.4),ylim=(1,5),legend=True,
               xlabel='$\phi$',ylabel='${\mu/\mu_0}$',show=False)
    p2=sp.plot(muM,(phi,0,0.4),ylim=(1,5),show=False)
    p1.append(p2[0])
    p1[0].label='Vand, fit valide environ jusqu''à $\phi$= 0.5'
    p1[0].line_color='red'
    p1[1].label='Mooney avec $\phi_0$ ='+str(PHI0)
    p1[1].line_color='black'
    p1.show()
    return
P1B3=interactive(f1B3interactif, PHI0=(0.3,1.2,0.1));

<>:19: SyntaxWarning: invalid escape sequence '\p'
<>:19: SyntaxWarning: invalid escape sequence '\m'
<>:22: SyntaxWarning: invalid escape sequence '\p'
<>:24: SyntaxWarning: invalid escape sequence '\p'
<>:19: SyntaxWarning: invalid escape sequence '\p'
<>:19: SyntaxWarning: invalid escape sequence '\m'
<>:22: SyntaxWarning: invalid escape sequence '\p'
<>:24: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_458619/918884941.py:19: SyntaxWarning: invalid escape sequence '\p'
  xlabel='$\phi$',ylabel='${\mu/\mu_0}$',show=False)
/tmp/ipykernel_458619/918884941.py:19: SyntaxWarning: invalid escape sequence '\m'
  xlabel='$\phi$',ylabel='${\mu/\mu_0}$',show=False)
/tmp/ipykernel_458619/918884941.py:22: SyntaxWarning: invalid escape sequence '\p'
  p1[0].label='Vand, fit valide environ jusqu''à $\phi$= 0.5'
/tmp/ipykernel_458619/918884941.py:24: SyntaxWarning: invalid escape sequence '\p'
  p1[1].label='Mooney avec $\phi_0$ ='+str(PHI0)


In [11]:
display(P1B3)

interactive(children=(FloatSlider(value=0.7, description='PHI0', max=1.2, min=0.3), Output()), _dom_classes=('…

### Section 1.7: Flux de quantité de mouvement.

>http://pierreproulx.espaceweb.usherbrooke.ca//images/GCH200_Ch1_resume.pdf